In [1]:
# Celda 1: ¿Qué Python estoy usando?
import sys
print(sys.executable)
print(sys.version)
# Debe mostrar: ...\.conda\envs\dab_project\python.exe
# Y Python 3.11.x

c:\Users\alexi\.conda\envs\dab_project\python.exe
3.11.15 | packaged by Anaconda, Inc. | (main, Mar 11 2026, 17:12:15) [MSC v.1942 64 bit (AMD64)]


In [2]:
# Celda 2: ¿databricks-connect está instalado y en qué versión?
import importlib.metadata
version = importlib.metadata.version("databricks-connect")
print(f"databricks-connect: {version}")
# Debe mostrar: 15.4.21

databricks-connect: 15.4.21


In [4]:
# Celda 3: ¿Los módulos core importan sin error?
from databricks.connect import DatabricksSession
from pyspark.sql import SparkSession
print("✅ Imports OK")

✅ Imports OK


In [3]:
# Celda 4: Ver TODOS los paquetes relevantes instalados
import importlib.metadata
paquetes = ["databricks-connect", "databricks-sdk", "pyspark", "numpy", "grpcio"]
for p in paquetes:
    try:
        v = importlib.metadata.version(p)
        print(f"✅ {p}: {v}")
    except importlib.metadata.PackageNotFoundError:
        print(f"❌ {p}: NO instalado")

✅ databricks-connect: 15.4.21
✅ databricks-sdk: 0.102.0
❌ pyspark: NO instalado
✅ numpy: 1.26.4
✅ grpcio: 1.78.0


In [5]:
import sys
import os

print("=== ENTORNO LOCAL ===")
print(f"Python: {sys.executable}")
print(f"Versión: {sys.version}")

# Verificar databricks-connect instalado
import importlib.metadata
print(f"\ndatabricks-connect: {importlib.metadata.version('databricks-connect')}")

# Verificar si existe configuración de Databricks
config_path = os.path.expanduser("~/.databrickscfg")
print(f"\n¿Existe ~/.databrickscfg? {os.path.exists(config_path)}")

if os.path.exists(config_path):
    with open(config_path) as f:
        # Solo muestra el perfil, no el token
        for line in f:
            if not "token" in line.lower():
                print(line.strip())

=== ENTORNO LOCAL ===
Python: c:\Users\alexi\.conda\envs\dab_project\python.exe
Versión: 3.11.15 | packaged by Anaconda, Inc. | (main, Mar 11 2026, 17:12:15) [MSC v.1942 64 bit (AMD64)]

databricks-connect: 15.4.21

¿Existe ~/.databrickscfg? True
[DEFAULT]
host  = https://dbc-6a5addb8-c11c.cloud.databricks.com/


In [9]:
from databricks.connect import DatabricksSession

# Para serverless se usa serverlessEnabled=True
spark = DatabricksSession.builder \
    .serverless(True) \
    .getOrCreate()

print("✅ Sesión serverless creada")
print(f"Spark version: {spark.version}")

✅ Sesión serverless creada
Spark version: 4.1.0


In [10]:
spark.sql("SHOW CATALOGS").show()

+---------+
|  catalog|
+---------+
|  samples|
|   system|
|workspace|
+---------+



In [11]:
from databricks.connect import DatabricksSession
from databricks.sdk.core import Config

# Opción A: usando perfil DEFAULT de ~/.databrickscfg
try:
    spark = DatabricksSession.builder.getOrCreate()
    print("✅ Conexión establecida")
    print(f"   Cluster ID: {spark.conf.get('spark.databricks.clusterUsageTags.clusterId', 'N/A')}")
except Exception as e:
    print(f"❌ Error de conexión: {e}")

✅ Conexión establecida
   Cluster ID: 0322-013320-hqgwznxx-v2n


In [7]:
# Si la celda 2 fue exitosa, esto lista los catálogos a los que tienes acceso
try:
    catalogos = spark.sql("SHOW CATALOGS").toPandas()
    print("✅ Catálogos disponibles:")
    print(catalogos)
except Exception as e:
    print(f"❌ Error: {e}")

✅ Catálogos disponibles:
     catalog
0    samples
1     system
2  workspace


In [14]:
# ✅ Forma correcta con databricks-connect
from databricks.sdk.runtime import dbutils

print("✅ dbutils importado correctamente")
print(type(dbutils))

✅ dbutils importado correctamente
<class 'databricks.sdk.dbutils.RemoteDbUtils'>


In [15]:
from databricks.sdk.runtime import dbutils

submódulos = ["fs", "secrets", "widgets", "notebook", "jobs"]

print("=== DISPONIBILIDAD DE DBUTILS ===")
for sub in submódulos:
    disponible = hasattr(dbutils, sub)
    estado = "✅" if disponible else "❌"
    print(f"{estado} dbutils.{sub}")

=== DISPONIBILIDAD DE DBUTILS ===
✅ dbutils.fs
✅ dbutils.secrets
✅ dbutils.widgets


c:\Users\alexi\.conda\envs\dab_project\Lib\site-packages\databricks\sdk\_widgets\__init__.py:71: UserWarning: 
To use databricks widgets interactively in your notebook, please install databricks sdk using:
	pip install 'databricks-sdk[notebook]'
Falling back to default_value_only implementation for databricks widgets.
  warnings.warn(


ValueError: cluster_id is required in the configuration. Config: host=https://dbc-6a5addb8-c11c.cloud.databricks.com, account_id=f8fa6054-2f74-48f4-9627-9372b82f9642, workspace_id=7474651262821619, discovery_url=https://dbc-6a5addb8-c11c.cloud.databricks.com/oidc/.well-known/oauth-authorization-server, auth_type=metadata-service, serverless_compute_id=auto, metadata_service_url=***. Env: DATABRICKS_HOST, DATABRICKS_AUTH_TYPE, DATABRICKS_SERVERLESS_COMPUTE_ID, DATABRICKS_METADATA_SERVICE_URL

In [16]:
from databricks.sdk.runtime import dbutils

# Listar archivos en DBFS
try:
    archivos = dbutils.fs.ls("/")
    print("✅ dbutils.fs.ls() funciona")
    print("\nContenido de DBFS raíz:")
    for f in archivos[:5]:  # solo primeros 5
        print(f"  {f.path} ({f.size} bytes)")
except Exception as e:
    print(f"❌ Error: {e}")

❌ Error: Public DBFS root is disabled. Access is denied on path: /. Config: host=https://dbc-6a5addb8-c11c.cloud.databricks.com, account_id=f8fa6054-2f74-48f4-9627-9372b82f9642, workspace_id=7474651262821619, discovery_url=https://dbc-6a5addb8-c11c.cloud.databricks.com/oidc/.well-known/oauth-authorization-server, auth_type=metadata-service, serverless_compute_id=auto, metadata_service_url=***. Env: DATABRICKS_HOST, DATABRICKS_AUTH_TYPE, DATABRICKS_SERVERLESS_COMPUTE_ID, DATABRICKS_METADATA_SERVICE_URL


In [17]:
from databricks.sdk.runtime import dbutils

# Listar scopes de secretos disponibles
try:
    scopes = dbutils.secrets.listScopes()
    print("✅ dbutils.secrets funciona")
    for s in scopes:
        print(f"  scope: {s.name}")
except Exception as e:
    print(f"❌ dbutils.secrets: {e}")

✅ dbutils.secrets funciona


In [8]:
# Reemplaza con tu catálogo, schema y tabla reales
CATALOG = "workspace"    # ← viene de tu databricks.yml
SCHEMA  = "default"      # ← viene de tu databricks.yml  
TABLA   = "nombre_tabla" # ← la tabla que quieres leer

try:
    df = spark.read.table(f"{CATALOG}.{SCHEMA}.{TABLA}")
    
    print(f"✅ Tabla leída: {CATALOG}.{SCHEMA}.{TABLA}")
    print(f"   Filas:    {df.count()}")
    print(f"   Columnas: {df.columns}")
    print("\n--- Primeras 5 filas ---")
    df.show(5, truncate=False)
    
except Exception as e:
    print(f"❌ Error leyendo tabla: {e}")

✅ Tabla leída: workspace.default.nombre_tabla
❌ Error leyendo tabla: [TABLE_OR_VIEW_NOT_FOUND] The table or view `workspace`.`default`.`nombre_tabla` cannot be found. Verify the spelling and correctness of the schema and catalog.
If you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.
To tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXISTS. SQLSTATE: 42P01;
'Aggregate [unresolvedalias(count(1))]
+- 'UnresolvedRelation [workspace, default, nombre_tabla], [], false


JVM stacktrace:
org.apache.spark.sql.catalyst.ExtendedAnalysisException
	at org.apache.spark.sql.catalyst.analysis.package$AnalysisErrorAt.tableNotFound(package.scala:94)
	at org.apache.spark.sql.catalyst.analysis.RelationResolution$.throwTableNotFound(RelationResolution.scala:1037)
	at org.apache.spark.sql.catalyst.analysis.CheckAnalysis.$anonfun$checkAnalysis0$2(CheckAnalysis.scala:355)
	at org.apache.spark.sql.catalys